# SC-3-Inheritance - Heritage et Interfaces

**Navigation** : [Index](../README.md) | [<< Functions](SC-2-Functions-State.ipynb) | [Errors >>](SC-4-Errors-Events.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**heritage** simple et multiple
2. Utiliser les **interfaces** pour l'abstraction
3. Maitriser **abstract** et **override**

### Duree estimee : 35 minutes

---

## 1. Heritage simple

Solidity supporte l'heritage avec le mot-cle `is`.

In [1]:
# Heritage simple
INHERITANCE_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

// Contrat parent
contract Animal {
    string public name;

    constructor(string memory _name) {
        name = _name;
    }

    function speak() public pure virtual returns (string memory) {
        return "...";
    }
}

// Contrat enfant
contract Dog is Animal {
    constructor(string memory _name) Animal(_name) {}

    function speak() public pure override returns (string memory) {
        return "Woof!";
    }
}

// Autre enfant
contract Cat is Animal {
    constructor(string memory _name) Animal(_name) {}

    function speak() public pure override returns (string memory) {
        return "Meow!";
    }
}
'''

print("Heritage simple avec override:")
print(INHERITANCE_BASIC)

Heritage simple avec override:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.30;

// Contrat parent
contract Animal {
    string public name;

    constructor(string memory _name) {
        name = _name;
    }

    function speak() public pure virtual returns (string memory) {
        return "...";
    }
}

// Contrat enfant
contract Dog is Animal {
    constructor(string memory _name) Animal(_name) {}

    function speak() public pure override returns (string memory) {
        return "Woof!";
    }
}

// Autre enfant
contract Cat is Animal {
    constructor(string memory _name) Animal(_name) {}

    function speak() public pure override returns (string memory) {
        return "Meow!";
    }
}



### 1.1 Ordre des constructeurs

In [2]:
# Ordre des constructeurs
CONSTRUCTOR_ORDER = '''
contract A {
    string public nameA;
    constructor(string memory _name) {
        nameA = _name;
    }
}

contract B is A {
    string public nameB;

    // Appel explicite du constructeur parent
    constructor(string memory _nameA, string memory _nameB) 
        A(_nameA)  // Constructeur parent d'abord
    {
        nameB = _nameB;
    }
}

// Sans arguments : constructeur par defaut
contract C is A {
    // Appel automatique de A("") si pas specifie
    constructor() A("Default") {}
}
'''

print("Ordre des constructeurs:")
print(CONSTRUCTOR_ORDER)

Ordre des constructeurs:

contract A {
    string public nameA;
    constructor(string memory _name) {
        nameA = _name;
    }
}

contract B is A {
    string public nameB;

    // Appel explicite du constructeur parent
    constructor(string memory _nameA, string memory _nameB) 
        A(_nameA)  // Constructeur parent d'abord
    {
        nameB = _nameB;
    }
}

// Sans arguments : constructeur par defaut
contract C is A {
    // Appel automatique de A("") si pas specifie
    constructor() A("Default") {}
}



---

## 2. Heritage multiple

Solidity permet l'heritage multiple avec resolution de linearisation (C3).

In [3]:
# Heritage multiple
MULTIPLE_INHERITANCE = '''
contract Ownable {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }

    function transferOwnership(address newOwner) public onlyOwner {
        owner = newOwner;
    }
}

contract Pausable {
    bool public paused = false;

    modifier whenNotPaused() {
        require(!paused, "Paused");
        _;
    }

    function pause() public {
        paused = true;
    }

    function unpause() public {
        paused = false;
    }
}

// Heritage multiple - ordre important!
// Le plus "de base" en premier
contract Token is Ownable, Pausable {
    string public name;
    uint256 public totalSupply;
    mapping(address => uint256) public balances;

    constructor(string memory _name, uint256 _supply) {
        name = _name;
        totalSupply = _supply;
        balances[msg.sender] = _supply;
    }

    function transfer(address to, uint256 amount) public whenNotPaused {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        balances[to] += amount;
    }

    function mint(address to, uint256 amount) public onlyOwner {
        totalSupply += amount;
        balances[to] += amount;
    }
}
'''

print("Heritage multiple avec modifiers herites:")
print(MULTIPLE_INHERITANCE)

Heritage multiple avec modifiers herites:

contract Ownable {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }

    function transferOwnership(address newOwner) public onlyOwner {
        owner = newOwner;
    }
}

contract Pausable {
    bool public paused = false;

    modifier whenNotPaused() {
        require(!paused, "Paused");
        _;
    }

    function pause() public {
        paused = true;
    }

    function unpause() public {
        paused = false;
    }
}

// Heritage multiple - ordre important!
// Le plus "de base" en premier
contract Token is Ownable, Pausable {
    string public name;
    uint256 public totalSupply;
    mapping(address => uint256) public balances;

    constructor(string memory _name, uint256 _supply) {
        name = _name;
        totalSupply = _supply;
        balances[msg.sender] = _supply;
    }

    function transfer(

---

## 3. Interfaces

Les interfaces definissent des signatures sans implementation.

In [4]:
# Interfaces
INTERFACE_EXAMPLE = '''
// Interface : que des signatures
interface IERC20 {
    function totalSupply() external view returns (uint256);
    function balanceOf(address account) external view returns (uint256);
    function transfer(address to, uint256 amount) external returns (bool);
    function allowance(address owner, address spender) external view returns (uint256);
    function approve(address spender, uint256 amount) external returns (bool);
    function transferFrom(address from, address to, uint256 amount) external returns (bool);

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
}

// Implementation de l'interface
contract SimpleToken is IERC20 {
    string public constant name = "Simple Token";
    string public constant symbol = "SIM";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _balances;
    mapping(address => mapping(address => uint256)) private _allowances;

    constructor(uint256 initialSupply) {
        _totalSupply = initialSupply;
        _balances[msg.sender] = initialSupply;
        emit Transfer(address(0), msg.sender, initialSupply);
    }

    function totalSupply() external view override returns (uint256) {
        return _totalSupply;
    }

    function balanceOf(address account) external view override returns (uint256) {
        return _balances[account];
    }

    function transfer(address to, uint256 amount) external override returns (bool) {
        _transfer(msg.sender, to, amount);
        return true;
    }

    function allowance(address owner, address spender) external view override returns (uint256) {
        return _allowances[owner][spender];
    }

    function approve(address spender, uint256 amount) external override returns (bool) {
        _allowances[msg.sender][spender] = amount;
        emit Approval(msg.sender, spender, amount);
        return true;
    }

    function transferFrom(address from, address to, uint256 amount) external override returns (bool) {
        require(_allowances[from][msg.sender] >= amount, "Insufficient allowance");
        _allowances[from][msg.sender] -= amount;
        _transfer(from, to, amount);
        return true;
    }

    function _transfer(address from, address to, uint256 amount) internal {
        require(_balances[from] >= amount, "Insufficient balance");
        _balances[from] -= amount;
        _balances[to] += amount;
        emit Transfer(from, to, amount);
    }
}
'''

print("Interface et implementation ERC-20:")
print(INTERFACE_EXAMPLE)

Interface et implementation ERC-20:

// Interface : que des signatures
interface IERC20 {
    function totalSupply() external view returns (uint256);
    function balanceOf(address account) external view returns (uint256);
    function transfer(address to, uint256 amount) external returns (bool);
    function allowance(address owner, address spender) external view returns (uint256);
    function approve(address spender, uint256 amount) external returns (bool);
    function transferFrom(address from, address to, uint256 amount) external returns (bool);

    event Transfer(address indexed from, address indexed to, uint256 value);
    event Approval(address indexed owner, address indexed spender, uint256 value);
}

// Implementation de l'interface
contract SimpleToken is IERC20 {
    string public constant name = "Simple Token";
    string public constant symbol = "SIM";
    uint8 public constant decimals = 18;

    uint256 private _totalSupply;
    mapping(address => uint256) private _ba

---

## 4. Contrats abstraits

Les contrats abstraits peuvent avoir des fonctions implementees ET non implementees.

In [5]:
# Contrats abstraits
ABSTRACT_EXAMPLE = '''
// Contrat abstrait : au moins une fonction sans implementation
abstract contract PriceFeed {
    // Fonction abstraite (doit etre implementee)
    function getPrice() public view virtual returns (uint256);

    // Fonction implementee (peut etre overridee)
    function getPriceWithDecimals() public view returns (uint256) {
        return getPrice() * 10 ** 18;
    }
}

// Implementation concrete
contract ChainlinkPriceFeed is PriceFeed {
    uint256 private price;

    constructor(uint256 _price) {
        price = _price;
    }

    function getPrice() public view override returns (uint256) {
        return price;
    }
}

// Implementation alternative
contract UniswapPriceFeed is PriceFeed {
    uint256 private price;

    constructor(uint256 _price) {
        price = _price;
    }

    function getPrice() public view override returns (uint256) {
        return price;
    }
    
    // Override de la fonction concrete
    function getPriceWithDecimals() public view override returns (uint256) {
        return getPrice() * 10 ** 6;  // USDC decimals
    }
}
'''

print("Contrats abstraits avec implementation partielle:")
print(ABSTRACT_EXAMPLE)

Contrats abstraits avec implementation partielle:

// Contrat abstrait : au moins une fonction sans implementation
abstract contract PriceFeed {
    // Fonction abstraite (doit etre implementee)
    function getPrice() public view virtual returns (uint256);

    // Fonction implementee (peut etre overridee)
    function getPriceWithDecimals() public view returns (uint256) {
        return getPrice() * 10 ** 18;
    }
}

// Implementation concrete
contract ChainlinkPriceFeed is PriceFeed {
    uint256 private price;

    constructor(uint256 _price) {
        price = _price;
    }

    function getPrice() public view override returns (uint256) {
        return price;
    }
}

// Implementation alternative
contract UniswapPriceFeed is PriceFeed {
    uint256 private price;

    constructor(uint256 _price) {
        price = _price;
    }

    function getPrice() public view override returns (uint256) {
        return price;
    }

    // Override de la fonction concrete
    function getPri

---

## 5. Super et linearisation

Le mot-cle `super` appelle la fonction du contrat parent.

In [6]:
# Super et linearisation
SUPER_EXAMPLE = '''
contract A {
    function foo() public pure virtual returns (string memory) {
        return "A";
    }
}

contract B is A {
    function foo() public pure virtual override returns (string memory) {
        return string.concat(super.foo(), " B");
    }
}

contract C is A {
    function foo() public pure virtual override returns (string memory) {
        return string.concat(super.foo(), " C");
    }
}

// Linearisation : D -> B -> C -> A
// super.foo() dans D appelle B.foo()
// super.foo() dans B appelle C.foo() (pas A directement!)
contract D is B, C {
    function foo() public pure override(B, C) returns (string memory) {
        return string.concat(super.foo(), " D");
    }
}
// Resultat: "A C B D"
'''

print("Linearisation C3 et super:")
print(SUPER_EXAMPLE)

Linearisation C3 et super:

contract A {
    function foo() public pure virtual returns (string memory) {
        return "A";
    }
}

contract B is A {
    function foo() public pure virtual override returns (string memory) {
        return string.concat(super.foo(), " B");
    }
}

contract C is A {
    function foo() public pure virtual override returns (string memory) {
        return string.concat(super.foo(), " C");
    }
}

// Linearisation : D -> B -> C -> A
// super.foo() dans D appelle B.foo()
// super.foo() dans B appelle C.foo() (pas A directement!)
contract D is B, C {
    function foo() public pure override(B, C) returns (string memory) {
        return string.concat(super.foo(), " D");
    }
}
// Resultat: "A C B D"



---

## 6. Exercices

### Exercice 1 : Contrat Vault

Creez un contrat Vault qui herite de Ownable et ajoute une fonction de depot/withdraw.

In [7]:
# Solution Exercice 1
SOLUTION_VAULT = '''
contract Ownable {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

contract Vault is Ownable {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
    }

    function withdrawAll() public onlyOwner {
        payable(owner).transfer(address(this).balance);
    }
}
'''
print("Solution Vault:")
print(SOLUTION_VAULT)

Solution Vault:

contract Ownable {
    address public owner;

    constructor() {
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }
}

contract Vault is Ownable {
    mapping(address => uint256) public balances;

    function deposit() public payable {
        balances[msg.sender] += msg.value;
    }

    function withdraw(uint256 amount) public {
        require(balances[msg.sender] >= amount, "Insufficient balance");
        balances[msg.sender] -= amount;
        payable(msg.sender).transfer(amount);
    }

    function withdrawAll() public onlyOwner {
        payable(owner).transfer(address(this).balance);
    }
}



### Exercice 2 : Interface IERC721 simplifiee

Creez une interface NFT simplifiee avec transfer et ownerOf.

In [8]:
# Solution Exercice 2
SOLUTION_NFT = '''
interface IERC721 {
    function ownerOf(uint256 tokenId) external view returns (address);
    function transferFrom(address from, address to, uint256 tokenId) external;
    event Transfer(address indexed from, address indexed to, uint256 indexed tokenId);
}

contract SimpleNFT is IERC721 {
    mapping(uint256 => address) private _owners;
    uint256 private _nextTokenId = 1;

    function mint() public returns (uint256) {
        uint256 tokenId = _nextTokenId++;
        _owners[tokenId] = msg.sender;
        emit Transfer(address(0), msg.sender, tokenId);
        return tokenId;
    }

    function ownerOf(uint256 tokenId) public view override returns (address) {
        require(_owners[tokenId] != address(0), "Token does not exist");
        return _owners[tokenId];
    }

    function transferFrom(address from, address to, uint256 tokenId) public override {
        require(_owners[tokenId] == from, "Not owner");
        _owners[tokenId] = to;
        emit Transfer(from, to, tokenId);
    }
}
'''
print("Solution NFT:")
print(SOLUTION_NFT)

Solution NFT:

interface IERC721 {
    function ownerOf(uint256 tokenId) external view returns (address);
    function transferFrom(address from, address to, uint256 tokenId) external;
    event Transfer(address indexed from, address indexed to, uint256 indexed tokenId);
}

contract SimpleNFT is IERC721 {
    mapping(uint256 => address) private _owners;
    uint256 private _nextTokenId = 1;

    function mint() public returns (uint256) {
        uint256 tokenId = _nextTokenId++;
        _owners[tokenId] = msg.sender;
        emit Transfer(address(0), msg.sender, tokenId);
        return tokenId;
    }

    function ownerOf(uint256 tokenId) public view override returns (address) {
        require(_owners[tokenId] != address(0), "Token does not exist");
        return _owners[tokenId];
    }

    function transferFrom(address from, address to, uint256 tokenId) public override {
        require(_owners[tokenId] == from, "Not owner");
        _owners[tokenId] = to;
        emit Transfer(fr

---

## 7. Resume

| Concept | Description |
|---------|-------------|
| `is` | Heritage d'un contrat parent |
| `virtual` | Fonction pouvant etre overidee |
| `override` | Redefinition d'une fonction parente |
| `interface` | Contrat 100% abstrait |
| `abstract` | Contrat partiellement implemente |
| `super` | Appel a la fonction parente |

---

**Notebook suivant** : [SC-4-Errors-Events](SC-4-Errors-Events.ipynb)